# 17. Working with Multiple Tables

*Module: 5 — Multi-table · Companion summary: `lectures/17-working-with-multiple-tables.md` · SQL engine: SQLite (in-memory)*

## What you will study

- filter with an IN (sub-query) across two tables
- answer the same question with a join instead — and compare
- see what a Cartesian product really produces
- write an implicit join with table aliases

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample data — EMPLOYEE + DEPARTMENT (the course's pair)

In [ ]:
q("CREATE TABLE emp (emp_id TEXT PRIMARY KEY, f_name TEXT, l_name TEXT, salary REAL, dep_id TEXT)")
q("""
INSERT INTO emp VALUES
    ('E1','Raul','Chong',95000,'D11'),
    ('E2','Rav','Ahuja',75000,'D11'),
    ('E3','Lakshmi','Katta',85000,'D20'),
    ('E4','Wei','Chen',60000,'D20'),
    ('E5','Ana','Silva',68000,'D30'),
    ('E6','Jack','Smith',55000,'D30')
""")
q("CREATE TABLE dept (dep_id TEXT PRIMARY KEY, dep_name TEXT, manager_id TEXT NOT NULL, loc_id TEXT)")
q("""
INSERT INTO dept VALUES
    ('D11','IT',30001,'L0001'),
    ('D20','HR',30002,'L0002'),
    ('D30','Sales',30003,'L0001')
""")
q("SELECT * FROM emp")
q("SELECT * FROM dept")

## 🧪 IN with an inner query — two-table lookup without a join

Employees who work in a department *located at L0002* — the sub-query turns the
departments table into a list of dep_ids:

In [ ]:
q("SELECT f_name, l_name, dep_id FROM emp WHERE dep_id IN (SELECT dep_id FROM dept WHERE loc_id = 'L0002')")

## 🧪 Same question, reversed — via join (lecture 18 preview)

In [ ]:
q("SELECT e.f_name, e.l_name, d.loc_id FROM emp e JOIN dept d ON e.dep_id = d.dep_id WHERE d.loc_id = 'L0002'")

## 🧪 “Who earns more than 70000?” — sub-query in the other direction

In [ ]:
q("SELECT dep_name FROM dept WHERE dep_id IN (SELECT dep_id FROM emp WHERE salary > 70000)")

## 🧪 Cartesian product — FORGETTING the join condition

In [ ]:
q("SELECT COUNT(*) FROM emp")
q("SELECT COUNT(*) FROM dept")
q("SELECT COUNT(*) FROM emp, dept")   # 6 x 3 = 18: every row pairs with every row


## 🧪 The fix — implicit join (comma syntax + WHERE), with aliases

In [ ]:
q("""
SELECT e.f_name, e.l_name, e.salary, d.dep_name, d.loc_id
FROM emp e, dept d
WHERE e.dep_id = d.dep_id
""")

## 💡 Tips & tricks

- Two tables, one FROM = Cartesian product **until** a WHERE matches keys — the classic
  accidental-row-explosion bug.
- Aliases (`emp e`) aren't just shorter: with the same table joined twice they're
  required (self-join).
- Same query, 2 spellings: `A, B WHERE keys match` ≈ `A JOIN B ON keys match` — the
  explicit JOIN is preferred (modern engines optimize both, but clarity wins).

## 🎯 Your turn

1. Department names of employees earning **less than 70000** (IN sub-query).
2. Same via the implicit-join spelling.
3. What's the Cartesian count for emp × emp (self-product)? Predict, then run.

In [ ]:
# 💡 Solution
q("SELECT dep_name FROM dept WHERE dep_id IN (SELECT dep_id FROM emp WHERE salary < 70000)")
q("SELECT DISTINCT d.dep_name FROM emp e, dept d WHERE e.dep_id = d.dep_id AND e.salary < 70000")
q("SELECT COUNT(*) AS cartesian FROM emp a, emp b")